# Sistema de Gestão de Eventos - Exemplo Consolidado

Este notebook é uma demonstração prática que consolida os conceitos avançados de **Pydantic** e **FastAPI**. O objetivo é gerir o ciclo de vida de eventos (Conferências, Workshops, etc.), aplicando validações rigorosas, lógica de negócio e serialização personalizada.

## 1. Importações e Configurações Globais

Importamos as bibliotecas necessárias e definimos expressões regulares para validar nomes de eventos e localizações.

In [1]:
import enum
import re
from datetime import datetime, timedelta
from typing import Any, Optional, TypeVar, Generic
from uuid import UUID, uuid4

from fastapi import FastAPI, HTTPException, status
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient
from pydantic import (
    BaseModel,
    EmailStr,
    Field,
    field_serializer,
    field_validator,
    model_validator,
    ConfigDict,
)

VALID_EVENT_NAME_REGEX = re.compile(r"^[a-zA-Z0-9\s]{5,100}$")
VALID_LOCATION_REGEX = re.compile(r"^[a-zA-Z0-9\s,.-]{5,200}$")

## 2. Definição de Enums

Utilizamos Enums para definir estados e categorias fixas, garantindo a integridade dos dados.

In [2]:
class EventStatus(enum.Enum):
    DRAFT = "draft"
    PUBLISHED = "published"
    CANCELLED = "cancelled"
    COMPLETED = "completed"

class EventCategory(enum.Enum):
    CONFERENCE = "conference"
    WORKSHOP = "workshop"
    MEETUP = "meetup"
    CONCERT = "concert"
    OTHER = "other"

## 3. Modelo Pydantic: Event

O modelo `Event` utiliza:
- **`ConfigDict`**: Para proibir campos extras e fornecer exemplos de documentação.
- **`Field`**: Com `default_factory` e `kw_only` para campos gerados automaticamente.
- **`field_validator`**: Para validação via Regex.
- **`model_validator(mode="after")`**: Para validar regras de negócio entre múltiplos campos (ex: datas).
- **`field_serializer`**: Para formatar a saída JSON.

In [3]:
class Event(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
        json_schema_extra={
            "examples": [
                {
                    "name": "Tech Conference 2026",
                    "description": "A conference about the latest in technology.",
                    "location": "Lisbon, Portugal",
                    "start_time": "2026-03-10T09:00:00Z",
                    "end_time": "2026-03-12T17:00:00Z",
                    "category": "conference",
                    "max_attendees": 500,
                    "created_by": "admin@example.com",
                }
            ]
        },
    )

    id: UUID = Field(default_factory=uuid4, description="Unique identifier", kw_only=True)
    name: str = Field(..., min_length=5, max_length=100)
    description: str = Field(..., min_length=10)
    location: str = Field(..., min_length=5, max_length=200)
    start_time: datetime
    end_time: datetime
    status: EventStatus = Field(default=EventStatus.DRAFT)
    category: EventCategory = Field(default=EventCategory.OTHER)
    max_attendees: int = Field(..., gt=0)
    created_by: EmailStr = Field(..., frozen=True)
    created_at: datetime = Field(default_factory=datetime.now, kw_only=True)

    @field_validator("name")
    @classmethod
    def validate_event_name(cls, v: str) -> str:
        if not VALID_EVENT_NAME_REGEX.match(v):
            raise ValueError("Event name is invalid.")
        return v

    @field_validator("location")
    @classmethod
    def validate_event_location(cls, v: str) -> str:
        if not VALID_LOCATION_REGEX.match(v):
            raise ValueError("Event location is invalid.")
        return v

    @model_validator(mode="after")
    def validate_times(self) -> 'Event':
        if self.start_time >= self.end_time:
            raise ValueError("Start time must be before end time.")
        return self

    @field_serializer("id", "created_by", when_used="json")
    def serialize_uuid_and_email(self, value: UUID | EmailStr) -> str:
        return str(value)

    @field_serializer("status", "category", when_used="json")
    def serialize_enum(self, value: EventStatus | EventCategory) -> str:
        return value.value

## 4. Resposta Genérica da API

Utilizamos `Generic` para padronizar as respostas da nossa API.

In [4]:
T = TypeVar('T')

class APIResponse(BaseModel, Generic[T]):
    status_code: int
    message: str
    data: Optional[T] = None

## 5. Aplicação FastAPI e Endpoints

Configuramos a aplicação e os endpoints CRUD para gerir os eventos.

In [5]:
app = FastAPI(title="Event Management System")
events_db: list[Event] = []

@app.post("/events", response_model=APIResponse[Event], status_code=status.HTTP_201_CREATED)
async def create_event(event: Event) -> APIResponse[Event]:
    events_db.append(event)
    return APIResponse(status_code=status.HTTP_201_CREATED, message="Event created successfully", data=event)

@app.get("/events", response_model=APIResponse[list[Event]])
async def list_events() -> APIResponse[list[Event]]:
    return APIResponse(status_code=status.HTTP_200_OK, message="Events retrieved successfully", data=events_db)

## 6. Testes e Demonstração

Executamos um teste de criação de evento para validar todo o sistema.

In [6]:
with TestClient(app) as client:
    print("Executando teste de criação de evento...")
    event_data = {
        "name": "Workshop de Python",
        "description": "Um workshop intensivo sobre programação.",
        "location": "Lisboa, Portugal",
        "start_time": (datetime.now() + timedelta(days=1)).isoformat(),
        "end_time": (datetime.now() + timedelta(days=1, hours=2)).isoformat(),
        "category": "workshop",
        "max_attendees": 50,
        "created_by": "instrutor@exemplo.com",
    }
    response = client.post("/events", json=event_data)
    if response.status_code == 201:
        print("Sucesso: Evento criado e validado pelo Pydantic!")
        print(f"Resposta JSON: {response.json()}")
    else:
        print(f"Erro na validação: {response.json()}")

Executando teste de criação de evento...
Sucesso: Evento criado e validado pelo Pydantic!
Resposta JSON: {'status_code': 201, 'message': 'Event created successfully', 'data': {'id': 'afbb0583-04f7-4190-9d09-158b82de7ca2', 'name': 'Workshop de Python', 'description': 'Um workshop intensivo sobre programação.', 'location': 'Lisboa, Portugal', 'start_time': '2026-03-05T02:41:56.263649', 'end_time': '2026-03-05T04:41:56.263670', 'status': 'draft', 'category': 'workshop', 'max_attendees': 50, 'created_by': 'instrutor@exemplo.com', 'created_at': '2026-03-04T02:41:56.356243'}}
